In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

/home/santripta/miniconda3/envs/llvis/lib/python3.13/site-packages/seaborn/_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.1)
  from scipy.stats import gaussian_kde


In [2]:
feats_split = pd.read_csv("../../experiment_data/qual_diff/feats_vol100_a3_resnet.csv")
points_split = pd.read_csv("../../experiment_data/qual_diff/points_vol100_a3_resnet.csv")

In [3]:
import pickle as pkl

In [4]:
from sklearn.cluster import AgglomerativeClustering

In [5]:
with open("clust_unnorm.pkl", "rb") as f:
    clust = pkl.load(f)

In [9]:
import networkx as nx

N = clust.n_leaves_
T = nx.DiGraph()

for (i, j) in clust.children_:
    T.add_node(i)
    T.add_node(j)

for i in range(N):
    if i not in T.nodes:
        T.add_node(i)
        print(f"Added leaf node {i} to tree")

for i, (c1, c2) in enumerate(clust.children_):
    id = i + N
    T.add_edge(id, c1)
    T.add_edge(id, c2)

In [ ]:
feats_interest = [(13,), (4,), (11,), (13, 4), (13, 11)]
ids = [feats_split[feats_split["Feature ID"].isin(fs)]["Feature ID"].unique() for fs in feats_interest]
points = [points_split[points_split["Feature ID"].isin(id_set)]["Data Index"].unique() for id_set in ids]

[len(p) for p in points]

[53, 253, 105, 306, 158]

In [13]:
def subcomp_size(node):
    desc = nx.descendants(T, node)
    leaves = [n for n in desc if n < N]
    return len(leaves)

def compute_subcomp_size(points):
    lcas = [nx.lowest_common_ancestor(T, points[0], points[1])]
    for p in points[2:]:
        lca = nx.lowest_common_ancestor(T, lcas[-1], p)
        lcas.append(lca)

    return subcomp_size(lcas[-1])

In [14]:
feat_sizes = [compute_subcomp_size(p) for p in points]
feat_sizes

[23673, 23673, 5587, 23673, 23673]

In [15]:
feats_interest_other = [(11, 4)]
ids_other = [feats_split[feats_split["Feature ID"].isin(fs)]["Feature ID"].unique() for fs in feats_interest_other]
points_other = [points_split[points_split["Feature ID"].isin(id_set)]["Data Index"].unique() for id_set in ids_other]

compute_subcomp_size(points_other[0])

23673